In [43]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error
from datetime import datetime, timedelta
import pytz

# =============================================
# ✅ إعداد المسارات
# =============================================
try:
    os.getcwd()
except FileNotFoundError:
    os.chdir(os.path.expanduser("~"))

# ✅ المسارات الرئيسية
BASE_DIR = '/Users/abdrhmn/projects/weather-data-pipeline'
ASSETS_DIR = os.path.join(BASE_DIR, 'assets')
WEATHER_CSV_PATH = os.path.join(ASSETS_DIR, 'weather.csv')

print(f"✅ مسار المشروع : {BASE_DIR}")
print(f"✅ مسار الملف   : {WEATHER_CSV_PATH}")
print(f"✅ الملف موجود؟ : {os.path.exists(WEATHER_CSV_PATH)}")

# =============================================
# تحميل API Key
# =============================================
env_path = os.path.join(os.path.expanduser("~"), '.env')
load_dotenv(env_path)
API_KEY = os.getenv("WEATHER_API_KEY")
BASE_URL = "https://api.openweathermap.org/data/2.5/"
print(f"✅ API Key جاهز: {API_KEY[:10]}...")

# =============================================
# جلب بيانات الطقس الحالية
# =============================================
def get_current_weather(city):
    try:
        url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return {
            'city': data['name'],  # ✅ إزالة المسافة
            'current_temp': round(data['main']['temp']),
            'feels_like': round(data['main']['feels_like']),
            'temp_min': round(data['main']['temp_min']),
            'temp_max': round(data['main']['temp_max']),
            'humidity': data['main']['humidity'],
            'description': data['weather'][0]['description'],
            'country': data['sys']['country'],
            'WindGustDir': data['wind'].get('deg', 0),  # ✅ اتجاه الريح بالدرجات
            'Pressure': data['main'].get('pressure', 0),
            'Wind_Gust_Speed': data['wind'].get('speed', 0),
            'WindGust': data['wind'].get('gust', None)
        }
    except Exception as e:
        print(f"❌ خطأ في جلب بيانات الطقس: {e}")
        return None

# =============================================
# قراءة البيانات التاريخية
# =============================================
def read_historical_data(filename):
    try:
        df = pd.read_csv(filename)
        print(f"✅ تم تحميل البيانات: {df.shape[0]} صف, {df.shape[1]} عمود")
        df = df.dropna()
        df = df.drop_duplicates()
        df = df.reset_index(drop=True)
        print(f"✅ بعد التنظيف: {df.shape[0]} صف")
        return df
    except FileNotFoundError:
        print(f"❌ لم يتم العثور على الملف: {filename}")
        raise

# =============================================
# تحضير البيانات للتدريب
# =============================================
def prepare_data(data):
    le = LabelEncoder()
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'].astype(str))
    data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'].astype(str))
    X = data.drop('RainTomorrow', axis=1)
    y = data['RainTomorrow']
    return X, y, le

# =============================================
# تدريب نموذج التنبؤ بالأمطار
# =============================================
def train_rain_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n📊 تقرير نموذج الأمطار:")
    print(classification_report(y_test, y_pred))
    return model

# =============================================
# تحضير بيانات الانحدار
# =============================================
def prepare_regression_data(data, feature):
    X, y = [], []
    values = data[feature].values
    for i in range(len(values) - 1):
        X.append(values[i])
        y.append(values[i + 1])
    return np.array(X).reshape(-1, 1), np.array(y)

# =============================================
# تدريب نموذج الانحدار
# =============================================
def train_regression_model(X, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model

# =============================================
# التنبؤ بالقيم المستقبلية
# =============================================
def predict_future(model, current_value, steps=5):
    predictions = [current_value]
    for _ in range(steps):
        next_value = model.predict(np.array([[predictions[-1]]]))[0]
        predictions.append(round(float(next_value), 2))
    return predictions[1:]  # ✅ إرجاع القيم المستقبلية فقط بدون القيمة الحالية

# =============================================
# الدالة الرئيسية
# =============================================
def weather_view():
    city = input("أدخل اسم المدينة: ")
    
    current_weather = get_current_weather(city)
    if not current_weather:
        print("❌ فشل جلب بيانات الطقس")
        return None
    
    # ✅ استخدام المسار الصحيح
    try:
        historical_data = read_historical_data(WEATHER_CSV_PATH)
    except FileNotFoundError:
        print(f"❌ لم يتم العثور على الملف في: {WEATHER_CSV_PATH}")
        return None
    
    X, y, le = prepare_data(historical_data)
    rain_model = train_rain_model(X, y)
    
    wind_deg = current_weather.get('WindGustDir', 0) % 360
    compass_points = [
        ("N", 348.75, 360), ("N", 0, 11.25),
        ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25),
        ("ESE", 101.25, 123.75), ("SE", 123.75, 146.25),
        ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25),
        ("WSW", 236.25, 258.75), ("W", 258.75, 281.25),
        ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75),
    ]
    
    compass_direction = "N"
    for direction, start, end in compass_points:
        if start <= wind_deg < end:
            compass_direction = direction
            break
    
    compass_direction_encoded = (
        le.transform([compass_direction])[0]
        if compass_direction in le.classes_ else 0
    )
    
    current_data = {
        'MinTemp': current_weather.get('temp_min', 0),
        'MaxTemp': current_weather.get('temp_max', 0),
        'Humidity': current_weather.get('humidity', 0),
        'WindGustDir': compass_direction_encoded,
        'WindGustSpeed': current_weather.get('Wind_Gust_Speed', 0),
        'Pressure': current_weather.get('Pressure', 0),
        'Temp': current_weather.get('current_temp', 0)
    }
    
    current_df = pd.DataFrame([current_data])
    
    try:
        rain_prediction = rain_model.predict(current_df)[0]
        rain_probability = (
            rain_model.predict_proba(current_df)[0][1] * 100
            if hasattr(rain_model, 'predict_proba')
            else float(rain_prediction) * 100
        )
    except Exception as e:
        print(f"⚠️ خطأ في التنبؤ بالأمطار: {e}")
        rain_probability = 0
    
    X_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
    X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')
    
    temp_model = train_regression_model(X_temp, y_temp)
    hum_model = train_regression_model(X_hum, y_hum)
    
    future_temp = predict_future(temp_model, current_weather.get('current_temp', 0))
    future_hum = predict_future(hum_model, current_weather.get('humidity', 0))
    
    # ✅ عرض النتائج بشكل صحيح
    print("\n" + "="*50)
    print(f"📍 تحليل الطقس لمدينة: {city}")
    print("="*50)
    print(f"🌡️  درجة الحرارة الحالية : {current_weather.get('current_temp')}°C")
    print(f"🌡️  أعلى درجة حرارة      : {current_weather.get('temp_max')}°C")
    print(f"🌡️  أدنى درجة حرارة      : {current_weather.get('temp_min')}°C")
    print(f"💧 الرطوبة               : {current_weather.get('humidity')}%")
    print(f"💨 اتجاه الريح           : {compass_direction}")
    print(f"🌧️  احتمالية الأمطار     : {rain_probability:.2f}%")
    print(f"📈 درجات الحرارة المتوقعة: {future_temp}")  # ✅ قائمة
    print(f"💧 الرطوبة المتوقعة      : {future_hum}")   # ✅ قائمة
    print("="*50)
    
    return {
        'city': city,
        'current_temp': current_weather.get('current_temp'),
        'temp_max': current_weather.get('temp_max'),
        'temp_min': current_weather.get('temp_min'),
        'humidity': current_weather.get('humidity'),
        'wind_direction': compass_direction,
        'rain_probability': rain_probability,
        'future_temp': future_temp,
        'future_humidity': future_hum
    }

# =============================================
# التشغيل الرئيسي
# =============================================
result = weather_view()

if result:
    timezone = pytz.timezone('Africa/Cairo')
    now = datetime.now(timezone)
    
    next_hour = (now + timedelta(hours=1)).replace(
        minute=0, second=0, microsecond=0
    )
    future_times = [next_hour + timedelta(hours=i) for i in range(5)]
    
    print(f"\n🕐 الوقت الحالي في القاهرة: {now.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"⏰ بداية الساعة القادمة  : {next_hour.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # ✅ عرض التوقعات مع الأوقات
    print("\n📅 التوقعات للساعات القادمة:")
    print("-"*55)
    print(f"{'الساعة':<10} {'الوقت':<22} {'الحرارة':^10} {'الرطوبة':^10}")
    print("-"*55)
    for i, (time, temp, hum) in enumerate(
        zip(future_times, result['future_temp'], result['future_humidity']), 1
    ):
        print(f"  {i:<8} {time.strftime('%Y-%m-%d %H:%M'):<22} {temp}°C{'':<6} {hum}%")
    
    print("\n📊 ملخص النتائج الكاملة:")
    print("-"*35)
    for key, value in result.items():
        print(f"  {key:<20}: {value}")
else:
    print("❌ لم تنجح عملية تحليل الطقس")

✅ مسار المشروع : /Users/abdrhmn/projects/weather-data-pipeline
✅ مسار الملف   : /Users/abdrhmn/projects/weather-data-pipeline/assets/weather.csv
✅ الملف موجود؟ : True
✅ API Key جاهز: c7f610680d...
✅ تم تحميل البيانات: 366 صف, 8 عمود
✅ بعد التنظيف: 363 صف

📊 تقرير نموذج الأمطار:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91        57
           1       0.86      0.38      0.52        16

    accuracy                           0.85        73
   macro avg       0.85      0.68      0.72        73
weighted avg       0.85      0.85      0.83        73

⚠️ خطأ في التنبؤ بالأمطار: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.


📍 تحليل الطقس لمدينة: England
🌡️  درجة الحرارة الحالية : 23°C
🌡️  أعلى درجة حرارة      : 23°C
🌡️  أدنى درجة حرارة      : 23°C
💧 الرطوبة               : 60%
💨 اتجاه الريح           : SSW
🌧️  احتمالية الأمطار     : 0.00%
📈 درجات الحرارة الم

/opt/miniconda3/envs/weather_project/lib/python3.11/site-packages/pytz/__init__.py:102: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
